# Import

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import math
import seaborn as sns
import pickle
import copy

import torch
from torch import nn, Tensor

import time
import joblib

from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='torch')

In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import fm.preprocessing as preProcess

In [3]:
# for the plots :
import mplhep as hep

plt.style.use(hep.style.CMS)

plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 13,

    "lines.linewidth": 2,

    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",

    "legend.frameon": False,
})


## Load the Datasets

In [37]:
# Inserire i nomi dei sample: 
sample_DY = "DYJetsToLL_M-50-madgraphMLM_btag_500k.npz"
sample_H = "GluGluHToTauTau_M125_btag_500k.npz"

In [38]:
data_DY = np.load(sample_DY)
data_H = np.load(sample_H)

## Prepare the DataFrames + Pre-Processing

In [39]:
df_DY = preProcess.build_df_from_npz_btag(data_DY, "DY", 0)
df_H  = preProcess.build_df_from_npz_btag(data_H,  "H",  1)

In [42]:
eps = 1e-8

for df_tmp in [df_DY, df_H]:

    for j in [1, 2, 3]:
        df_tmp[f"jet{j}_logpt"] = np.log(
            np.clip(df_tmp[f"jet{j}_pt"], eps, None)
        )

In [43]:
eps = 1e-8

for df_tmp in [df_DY, df_H]:
    
    # log pT tau reco
    df_tmp["tau1_logpt"] = np.log(np.clip(df_tmp["tau1_pt"], eps, None))
    df_tmp["tau2_logpt"] = np.log(np.clip(df_tmp["tau2_pt"], eps, None))

    # log pT tau gen
    df_tmp["tau1_gen_logpt"] = np.log(np.clip(df_tmp["tau1_gen_pt"], eps, None))
    df_tmp["tau2_gen_logpt"] = np.log(np.clip(df_tmp["tau2_gen_pt"], eps, None))

    # pT reco
    df_tmp["tau1_pt_reco"] = df_tmp["tau1_pt"]
    df_tmp["tau2_pt_reco"] = df_tmp["tau2_pt"]

    # pT reco corretto PNet
    df_tmp["tau1_pt_reco_corrPNet"] = (
        df_tmp["tau1_pt_reco"] * df_tmp["tau1_ptCorrPNet"]
    )
    df_tmp["tau2_pt_reco_corrPNet"] = (
        df_tmp["tau2_pt_reco"] * df_tmp["tau2_ptCorrPNet"]
    )

    # correzioni target/reco_corrPNet
    df_tmp["tau1_corr"] = (
        df_tmp["tau1_gen_pt"] / np.clip(df_tmp["tau1_pt_reco_corrPNet"], eps, None)
    )
    df_tmp["tau2_corr"] = (
        df_tmp["tau2_gen_pt"] / np.clip(df_tmp["tau2_pt_reco_corrPNet"], eps, None)
    )

    # log pT dei jet, richiesto da apply_jet_selection
    for j in [1, 2, 3]:
        df_tmp[f"jet{j}_logpt"] = np.log(
            np.clip(df_tmp[f"jet{j}_pt"], eps, None)
        )

    # rimuovo eventi con correzione PNet nulla/non fisica
    mask = (
        (df_tmp["tau1_pt_reco_corrPNet"] > 0) &
        (df_tmp["tau2_pt_reco_corrPNet"] > 0)
    )

    df_tmp.drop(index=df_tmp.index[~mask], inplace=True)

In [45]:
# pT cut

pt_cfg = preProcess.PtCutConfig(
    pt_min=30.0,
    space="pt",
    tau1_pt_col="tau1_pt_reco",
    tau2_pt_col="tau2_pt_reco",
)

df_DY_pt = preProcess.apply_pt_cut(df_DY, pt_cfg)
df_H_pt  = preProcess.apply_pt_cut(df_H,  pt_cfg)

print("DY :", len(df_DY), "->", len(df_DY_pt))
print("H  :", len(df_H),  "->", len(df_H_pt))

DY : 499850 -> 102972
H  : 499910 -> 222915


In [46]:
# Jet selection 

df_DY_pt_jets = preProcess.apply_jet_selection(df_DY_pt, pt_min_jet=20.0)
df_H_pt_jets  = preProcess.apply_jet_selection(df_H_pt,  pt_min_jet=20.0)

cutflow_df = pd.DataFrame({
    "step": ["raw", "tau_pt", "jets"],
    "DY": [len(df_DY), len(df_DY_pt), len(df_DY_pt_jets)],
    "H":  [len(df_H),  len(df_H_pt),  len(df_H_pt_jets)],
})

cutflow_df

,step,DY,H
0,raw,499850,499910
1,tau_pt,102972,222915
2,jets,26118,85977


In [47]:
# valid taus
tauid_cfg = preProcess.TauIDConfig(
    tau1_id_col="tau1_rawPNetVSjet",
    tau2_id_col="tau2_rawPNetVSjet",
    invalid_value=-1.0,
)

df_DY_valid = preProcess.filter_valid_tauid(df_DY_pt_jets, tauid_cfg)
df_H_valid  = preProcess.filter_valid_tauid(df_H_pt_jets,  tauid_cfg)

# check
print("DY after jets:", len(df_DY_pt_jets), "-> after valid tauID:", len(df_DY_valid))
print("H  after jets:", len(df_H_pt_jets),  "-> after valid tauID:", len(df_H_valid))

assert (df_DY_valid["tau1_rawPNetVSjet"] != -1).all()
assert (df_DY_valid["tau2_rawPNetVSjet"] != -1).all()

assert (df_H_valid["tau1_rawPNetVSjet"] != -1).all()
assert (df_H_valid["tau2_rawPNetVSjet"] != -1).all()

print("valid tauID OK (no -1)")

DY after jets: 26118 -> after valid tauID: 25828
H  after jets: 85977 -> after valid tauID: 85491
valid tauID OK (no -1)


In [48]:
# calcolo sulle soglie:
tauid_cfg = preProcess.TauIDConfig(
    tau1_id_col="tau1_rawPNetVSjet",
    tau2_id_col="tau2_rawPNetVSjet",
    invalid_value=-1.0,
    wp_mode="target_eff",
    target_eff=0.90,
    reference="higgs",
    require_both=True,
)

# soglie:
thr1, thr2 = preProcess.compute_tauid_thresholds(df_DY_valid, df_H_valid, tauid_cfg)
print("thr_tau1 =", thr1)
print("thr_tau2 =", thr2)

# applico WP:
df_DY_tauid = preProcess.apply_tauid_wp(df_DY_valid, tauid_cfg, thr1, thr2)
df_H_tauid  = preProcess.apply_tauid_wp(df_H_valid,  tauid_cfg, thr1, thr2)

thr_tau1 = 0.638671875
thr_tau2 = 0.470458984375


In [49]:
cutflow_df = pd.DataFrame({
    "step": ["raw", "tau_pt", "jets", "valid_tauID", "tauID_WP"],
    "DY": [
        len(df_DY),
        len(df_DY_pt),
        len(df_DY_pt_jets),
        len(df_DY_valid),
        len(df_DY_tauid),
    ],
    "H": [
        len(df_H),
        len(df_H_pt),
        len(df_H_pt_jets),
        len(df_H_valid),
        len(df_H_tauid),
    ],
})

cutflow_df

,step,DY,H
0,raw,499850,499910
1,tau_pt,102972,222915
2,jets,26118,85977
3,valid_tauID,25828,85491
4,tauID_WP,18836,69370


In [50]:
print("DY final efficiency:", len(df_DY_tauid) / len(df_DY))
print("H final efficiency :", len(df_H_tauid) / len(df_H))

DY final efficiency: 0.03768330499149745
H final efficiency : 0.13876497769598528


In [51]:
cutflow_df["DY_eff_step"] = cutflow_df["DY"] / cutflow_df["DY"].shift(1)
cutflow_df["H_eff_step"]  = cutflow_df["H"]  / cutflow_df["H"].shift(1)

cutflow_df["DY_eff_raw"] = cutflow_df["DY"] / cutflow_df["DY"].iloc[0]
cutflow_df["H_eff_raw"]  = cutflow_df["H"]  / cutflow_df["H"].iloc[0]

cutflow_df

,step,DY,H,DY_eff_step,H_eff_step,DY_eff_raw,H_eff_raw
0,raw,499850,499910,NaN,NaN,1.000000,1.000000
1,tau_pt,102972,222915,0.206006,0.445910,0.206006,0.445910
2,jets,26118,85977,0.253642,0.385694,0.052252,0.171985
3,valid_tauID,25828,85491,0.988897,0.994347,0.051672,0.171013
4,tauID_WP,18836,69370,0.729286,0.811430,0.037683,0.138765


In [53]:
df_DY_event = df_DY_tauid.copy()
df_H_event = df_H_tauid.copy()

In [54]:
df_DY_event.to_pickle("df_DY_event+btag.pkl")
df_H_event.to_pickle("df_H_event+btag.pkl")